# Energy-Based Models for Credit Card Fraud Detection

This notebook demonstrates using an EBM trained from scratch for anomaly detection on the Credit Card Fraud dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. Load Credit Card Fraud Dataset

The dataset contains 284,807 transactions with 492 frauds (0.17%). Features V1-V28 are PCA-transformed (for privacy), plus Amount.

In [ ]:
from ebm.anomaly.data import load_credit_card_data, get_dataset_summary

dataset = load_credit_card_data(
    path="creditcard.csv",
    train_ratio=0.8,
    val_ratio=0.1,
    normalize=True,
    random_state=42
)

print(get_dataset_summary(dataset))

In [ ]:
print(f"Feature dimensions: {dataset.X_train.shape[1]}")
print(f"Feature names: {dataset.feature_names[:5]} ... {dataset.feature_names[-2:]}")
print("\nTraining data stats:")
print(f"  Mean: {dataset.X_train.mean():.4f}")
print(f"  Std:  {dataset.X_train.std():.4f}")
print(f"  Min:  {dataset.X_train.min():.4f}")
print(f"  Max:  {dataset.X_train.max():.4f}")

## 2. Create Energy-Based Model

The energy function assigns low energy to normal transactions and high energy to anomalies.

In [ ]:
from ebm.core.energy import EnergyMLP
from ebm.training.optimizer import AdamW
from ebm.sampling.replay_buffer import ReplayBuffer

input_dim = dataset.X_train.shape[1]
energy_fn = EnergyMLP(
    input_dim=input_dim,
    hidden_dims=[128, 128],
    activation="swish"
)

print(energy_fn)
print(f"\nTotal parameters: {sum(p.size for p in energy_fn.parameters()):,}")

In [ ]:
optimizer = AdamW(
    energy_fn.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

buffer = ReplayBuffer(
    capacity=10000,
    sample_dim=input_dim,
    init_type="gaussian",
    init_std=1.0
)

print(f"Replay buffer: {buffer}")

## 3. Train the EBM

We train using contrastive divergence:
- Push DOWN energy of real (normal) data
- Push UP energy of samples from Langevin dynamics

In [ ]:
from ebm.training.trainer import train

train_subset = dataset.X_train[:50000]

print(f"Training on {len(train_subset):,} samples...")
print("="*60)

In [ ]:
# Train the model
history = train(
    energy_fn=energy_fn,
    optimizer=optimizer,
    dataset=train_subset,
    n_epochs=20,
    batch_size=128,
    replay_buffer=buffer,
    langevin_steps=20,
    langevin_step_size=0.01,
    langevin_noise_scale=1.0,
    langevin_grad_clip=0.03,
    alpha=0.1,
    reinit_prob=0.05,
    compute_entropy=False,
    verbose=True,
    log_interval=2
)

print("="*60)
print(f"Training complete! Best loss: {history.best_loss:.4f} at epoch {history.best_epoch}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.get_metric('loss'))
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.get_metric('E_real'), label='E(normal)', color='green')
axes[1].plot(history.get_metric('E_fake'), label='E(samples)', color='red')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Energy')
axes[1].set_title('Energy Values')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Create Anomaly Detector

Use the trained energy function to detect anomalies:
- High energy = anomaly (fraud)
- Low energy = normal

In [ ]:
from ebm.anomaly.detector import EBMAnomalyDetector

detector = EBMAnomalyDetector(energy_fn)

threshold = detector.fit_threshold(
    X_val=dataset.X_val,
    y_val=dataset.y_val,
    percentile=95
)

print(f"Fitted threshold: {threshold:.4f}")
print("\nThreshold info:")
print(detector.threshold_info)

## 5. Evaluate on Test Set

In [ ]:
from ebm.anomaly.evaluate import (
    evaluate_detector,
    compute_auroc,
    compute_auprc,
    format_evaluation_report
)

results = evaluate_detector(
    detector=detector,
    X_test=dataset.X_test,
    y_test=dataset.y_test
)

print(format_evaluation_report(results))

In [ ]:
test_scores = detector.score(dataset.X_test)

auroc = compute_auroc(dataset.y_test, test_scores)
auprc = compute_auprc(dataset.y_test, test_scores)

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")
print(f"\n(Random baseline AUROC = 0.5, AUPRC = {dataset.y_test.mean():.4f})")

## 6. Visualize Results

In [ ]:
normal_scores = test_scores[dataset.y_test == 0]
fraud_scores = test_scores[dataset.y_test == 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(normal_scores, bins=50, alpha=0.7, label=f'Normal (n={len(normal_scores)})', density=True)
axes[0].hist(fraud_scores, bins=50, alpha=0.7, label=f'Fraud (n={len(fraud_scores)})', density=True)
axes[0].axvline(threshold, color='red', linestyle='--', label=f'Threshold={threshold:.2f}')
axes[0].set_xlabel('Energy (Anomaly Score)')
axes[0].set_ylabel('Density')
axes[0].set_title('Energy Distribution: Normal vs Fraud')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].boxplot([normal_scores, fraud_scores], labels=['Normal', 'Fraud'])
axes[1].axhline(threshold, color='red', linestyle='--', label='Threshold')
axes[1].set_ylabel('Energy (Anomaly Score)')
axes[1].set_title('Energy Distribution Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Normal energy: {normal_scores.mean():.2f} +/- {normal_scores.std():.2f}")
print(f"Fraud energy:  {fraud_scores.mean():.2f} +/- {fraud_scores.std():.2f}")
print(f"Separation:    {fraud_scores.mean() - normal_scores.mean():.2f}")

In [ ]:
from ebm.anomaly.evaluate import roc_curve, precision_recall_curve

fpr, tpr, _ = roc_curve(dataset.y_test, test_scores)
precision, recall, _ = precision_recall_curve(dataset.y_test, test_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'EBM (AUROC={auroc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random (AUROC=0.5)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

baseline = dataset.y_test.mean()
axes[1].plot(recall, precision, 'b-', linewidth=2, label=f'EBM (AUPRC={auprc:.3f})')
axes[1].axhline(baseline, color='k', linestyle='--', label=f'Random (AUPRC={baseline:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 7. Confusion Matrix Analysis

In [ ]:
from ebm.anomaly.evaluate import confusion_matrix

predictions = detector.predict(dataset.X_test)

res = confusion_matrix(dataset.y_test, predictions)

print("Confusion Matrix:")
print("="*40)
print("                  Predicted")
print("                Normal  Fraud")
print(f"Actual Normal    {res['TN']:6d}  {res['FP']:5d}")
print(f"Actual Fraud     {res['FN']:6d}  {res['TP']:5d}")
print("="*40)
print(f"\nTrue Negatives:  {res['TN']:,} (correctly identified normal)")
print(f"False Positives: {res['FP']:,} (normal flagged as fraud)")
print(f"False Negatives: {res['FN']:,} (fraud missed)")
print(f"True Positives:  {res['TP']:,} (fraud caught)")
print(f"\nFraud Detection Rate: {res['TP']/(res['TP']+res['FN'])*100:.1f}% ({res['TP']} of {res['TP']+res['FN']} frauds caught)")

## Summary

This notebook demonstrated:

1. **Data Loading** - Semi-supervised setup (train on normal only)
2. **EBM Architecture** - MLP energy function with Swish activation  
3. **Training** - Contrastive divergence with Langevin sampling
4. **Anomaly Detection** - High energy = anomaly
5. **Evaluation** - AUROC, AUPRC, precision/recall, confusion matrix

**Key insight**: The EBM learns to assign low energy to the normal transaction manifold. Fraudulent transactions lie outside this manifold and receive higher energy.

**To improve results**:
- Train on full dataset with more epochs
- Tune hyperparameters (learning rate, architecture, Langevin steps)
- Use larger hidden dimensions
- Add entropy regularization